# DataSet and MappedDataSet are the same tokens

Two artifacts over the same tokenizer and the same ordered sources. `DataSet`'s
job physically concatenates every source's `tokens.bin` into its own
`train.bin`/`valid.bin`, with `<|endoftext|>` between sources, and binding maps
those files. `MappedDataSet` writes nothing: binding opens each source's own
`tokens.bin` and serves the same logical concatenation, separator included,
through one index. This notebook builds both against a throwaway root and
checks they are bit identical.

In [1]:
import logging
import shutil
from pathlib import Path

import numpy as np

import lab
from artifacts.core.resolve import resolve
from artifacts.dataset import DataSet
from artifacts.mappeddataset import MappedDataSet
from artifacts.sources import SourceURL
from artifacts.tokenizers.bpe import Tokenizer

ROOT = Path(".scratch/dataset-identity").resolve()
shutil.rmtree(ROOT, ignore_errors=True)
ROOT.mkdir(parents=True)
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
ROOT

PosixPath('/Users/oguz/Projects/launchpad/.scratch/dataset-identity')

In [2]:
romeojuliet = SourceURL(name="romeojuliet", url="https://www.gutenberg.org/cache/epub/1513/pg1513.txt")
modestproposal = SourceURL(name="modestproposal", url="https://www.gutenberg.org/cache/epub/1080/pg1080.txt")

tokenizer = Tokenizer(vocab_size=400, special_tokens=("<|endoftext|>",), sources=(romeojuliet, modestproposal))

train, valid = (romeojuliet, modestproposal), (modestproposal,)
dataset = DataSet.from_sources(tokenizer, train_sources=train, valid_sources=valid)
mapped = MappedDataSet.from_sources(tokenizer, train_sources=train, valid_sources=valid)

print("dataset:", dataset.artifact_path, "producer:", dataset.producer)
print("mapped: ", mapped.artifact_path, "producer:", mapped.producer)

dataset: datasets/dataset-1d4dacfcc4 producer: artifacts.dataset.jobs.DataSetJob
mapped:  mappeddatasets/mapped-1d4dacfcc4 producer: None


## Build

Both share every dependency (sources, tokenizer, tokenized sources), so the
second declaration only adds its own manifest. The mapped dataset has no job
to run; the dataset's job runs last, once every `tokens.bin` is there.

In [3]:
lab.declare(dataset, root=ROOT, commit=True)
lab.declare(mapped, root=ROOT, commit=True)

for artifact in resolve(dataset):
    if artifact.producer is None or all(artifact.status(ROOT).completion.values()):
        continue
    print(f"running {artifact.producer} for {artifact.artifact_path}")
    artifact.job().run(ROOT, lab.worker)

lab.declare(mapped, root=ROOT)  # done without a job of its own

sources/romeojuliet                          declared  (created)
sources/modestproposal                       declared  (created)
tokenizers/bpe-400-06574802d7                declared  (created)
tokenized/bpe-400-06574802d7/romeojuliet     declared  (created)
tokenized/bpe-400-06574802d7/modestproposal  declared  (created)
datasets/dataset-1d4dacfcc4                  declared  (created)

6 declared
6 manifests published: sources/romeojuliet, sources/modestproposal, tokenizers/bpe-400-06574802d7, tokenized/bpe-400-06574802d7/romeojuliet, tokenized/bpe-400-06574802d7/modestproposal, datasets/dataset-1d4dacfcc4
sources/romeojuliet                          declared
sources/modestproposal                       declared
tokenizers/bpe-400-06574802d7                declared
tokenized/bpe-400-06574802d7/romeojuliet     declared
tokenized/bpe-400-06574802d7/modestproposal  declared
mappeddatasets/mapped-1d4dacfcc4             declared  (created)

6 declared
1 manifest published: mappeddatasets/

/Users/oguz/Projects/launchpad/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
downloading romeojuliet from https://www.gutenberg.org/cache/epub/1513/pg1513.txt
downloading romeojuliet from https://www.gutenberg.org/cache/epub/1513/pg1513.txt
wrote 167469 chars for romeojuliet
wrote 167469 chars for romeojuliet
downloading modestproposal from https://www.gutenberg.org/cache/epub/1080/pg1080.txt
downloading modestproposal from https://www.gutenberg.org/cache/epub/1080/pg1080.txt


running artifacts.sources.jobs.SourceURLJob for sources/modestproposal


wrote 39525 chars for modestproposal
wrote 39525 chars for modestproposal
training BPE tokenizer (vocab_size=400) on 2 source(s)
training BPE tokenizer (vocab_size=400) on 2 source(s)


running artifacts.tokenizers.bpe.jobs.TokenizerJob for tokenizers/bpe-400-06574802d7


trained, vocab has 400 entries
trained, vocab has 400 entries
tokenizing romeojuliet
tokenizing romeojuliet
wrote 95516 tokens for romeojuliet
wrote 95516 tokens for romeojuliet
tokenizing modestproposal
tokenizing modestproposal
wrote 21653 tokens for modestproposal
wrote 21653 tokens for modestproposal
building training set from 2 source(s)
building training set from 2 source(s)
building validation set from 1 source(s)
building validation set from 1 source(s)
done
done


running artifacts.tokenized.jobs.TokenizeSourceJob for tokenized/bpe-400-06574802d7/romeojuliet
running artifacts.tokenized.jobs.TokenizeSourceJob for tokenized/bpe-400-06574802d7/modestproposal
running artifacts.dataset.jobs.DataSetJob for datasets/dataset-1d4dacfcc4
sources/romeojuliet                          done
sources/modestproposal                       done
tokenizers/bpe-400-06574802d7                done
tokenized/bpe-400-06574802d7/romeojuliet     done
tokenized/bpe-400-06574802d7/modestproposal  done
mappeddatasets/mapped-1d4dacfcc4             done

6 done
ok -- 0 to declare


DeclarationReport(rows=[{'path': 'sources/romeojuliet', 'state': 'done', 'drift': False, 'differences': {}, 'created': False, 'commit': ['2547940abe0ab7248b89164783cbe4c0834b551d-dirty', '2547940abe0ab7248b89164783cbe4c0834b551d-dirty']}, {'path': 'sources/modestproposal', 'state': 'done', 'drift': False, 'differences': {}, 'created': False, 'commit': ['2547940abe0ab7248b89164783cbe4c0834b551d-dirty', '2547940abe0ab7248b89164783cbe4c0834b551d-dirty']}, {'path': 'tokenizers/bpe-400-06574802d7', 'state': 'done', 'drift': False, 'differences': {}, 'created': False, 'commit': ['2547940abe0ab7248b89164783cbe4c0834b551d-dirty', '2547940abe0ab7248b89164783cbe4c0834b551d-dirty']}, {'path': 'tokenized/bpe-400-06574802d7/romeojuliet', 'state': 'done', 'drift': False, 'differences': {}, 'created': False, 'commit': ['2547940abe0ab7248b89164783cbe4c0834b551d-dirty', '2547940abe0ab7248b89164783cbe4c0834b551d-dirty']}, {'path': 'tokenized/bpe-400-06574802d7/modestproposal', 'state': 'done', 'drift': 

## Bind and compare

`bind(root)` hands each back with `train_tokens`/`valid_tokens` loaded:
`np.memmap`s over `train.bin`/`valid.bin` for the dataset, a `TokenStream` per
split for the mapped one. Same length, same dtype, same token at every
position, and the mapped stream's bytes are exactly the file the dataset job
wrote.

In [4]:
dataset = dataset.bind(ROOT)
mapped = mapped.bind(ROOT)

for split in ("train_tokens", "valid_tokens"):
    physical, virtual = getattr(dataset, split), getattr(mapped, split)
    assert len(physical) == len(virtual), (len(physical), len(virtual))
    assert physical.dtype == virtual.dtype == np.uint16
    assert np.array_equal(physical[:], virtual[:])
    print(f"{split}: {len(physical)} tokens, identical")

assert dataset.paths(ROOT)["training set"].read_bytes() == mapped.train_tokens[:].tobytes()
assert dataset.paths(ROOT)["validation set"].read_bytes() == mapped.valid_tokens[:].tobytes()
print("train.bin and valid.bin are byte-for-byte what the mapped stream serves")

train_tokens: 117170 tokens, identical
valid_tokens: 21653 tokens, identical
train.bin and valid.bin are byte-for-byte what the mapped stream serves


## Windows across the boundary

The interesting positions are around the seam between the two training
sources: the dataset job wrote a `<|endoftext|>` there, and the mapped stream
synthesizes one at the same index. Random windows anywhere, boundary included,
agree, and a window inside one source is still that source's own memmap.

In [5]:
bound_tokenizer = tokenizer.bind(ROOT)
[eot] = bound_tokenizer.encode("<|endoftext|>")
seam = mapped.train_set[0].paths(ROOT)["tokens"].stat().st_size // 2  # first source's token count

assert dataset.train_tokens[seam] == mapped.train_tokens[seam] == eot
print(f"token {seam} is <|endoftext|> ({eot}) in both")
print(repr(bound_tokenizer.decode(mapped.train_tokens[seam - 8 : seam + 8])))

rng = np.random.default_rng(0)
block = 64
for start in rng.integers(0, len(mapped.train_tokens) - block, size=1000):
    assert np.array_equal(dataset.train_tokens[start : start + block], mapped.train_tokens[start : start + block])
assert np.array_equal(dataset.train_tokens[seam - 3 : seam + 3], mapped.train_tokens[seam - 3 : seam + 3])
assert isinstance(mapped.train_tokens[:block], np.memmap)
assert not isinstance(mapped.train_tokens[seam - 3 : seam + 3], np.memmap)  # stitched: a copy of the window
print("1000 random windows agree; a window inside one source is a zero-copy memmap slice")

token 95516 is <|endoftext|> (399) in both
'Books.\n\n\n<|endoftext|>The Project Gutenberg eB'
1000 random windows agree; a window inside one source is a zero-copy memmap slice


In [6]:
mapped.train_tokens

In [7]:
# Cleanup, if you want the demo folder gone:
# shutil.rmtree(ROOT)